# Income Prediction Using TensorFlow and PyTorch
**Dataset:** Adult Census Income (`adult_dataset.csv`)  
**Target:** `income` (`<=50K` vs `>50K`)  
**Goal:** Explore and preprocess the data, train comparable neural-network classifiers in TensorFlow and PyTorch, evaluate them, and optionally tune hyperparameters with Optuna.

> Run this notebook in **Google Colab**. Upload `adult_dataset.csv` when prompted. The notebook prints actual evaluation results when executed; do not report results before running it.

## 1. Install / import libraries
TensorFlow, PyTorch, pandas, scikit-learn and matplotlib are commonly available in Colab. Optuna is installed for the optional tuning section.

In [ ]:
!pip -q install optuna

import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)

import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("TensorFlow:", tf.__version__)
print("PyTorch:", torch.__version__)
print("PyTorch device:", "cuda" if torch.cuda.is_available() else "cpu")

## 2. Load the dataset
In Colab, choose the CSV file from your computer. If the file is already in the current working directory, the upload prompt is skipped.

In [ ]:
DATA_PATH = "adult_dataset.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            DATA_PATH = next(iter(uploaded.keys()))
    except Exception:
        pass

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())

## 3. Exploratory Data Analysis (EDA)
Check data types, missing values, duplicates, target distribution, and basic numeric summaries.

In [ ]:
display(df.info())
display(df.describe(include="all").T)
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values (top):")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

if "income" not in df.columns:
    raise ValueError("Expected target column 'income' was not found. Check the CSV column names.")

plt.figure(figsize=(6, 4))
df["income"].astype(str).value_counts().plot(kind="bar")
plt.title("Income target distribution")
plt.xlabel("Income class")
plt.ylabel("Number of rows")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

numeric_for_plot = df.select_dtypes(include=np.number).columns.tolist()
if numeric_for_plot:
    df[numeric_for_plot].hist(figsize=(14, 9), bins=25)
    plt.suptitle("Numeric feature distributions")
    plt.tight_layout()
    plt.show()

## 4. Clean and preprocess
Some Adult dataset versions represent missing categorical values with `?` or spaces. Convert these to missing values. Remove exact duplicate rows. Split the target from features, then split into training and test sets. The imputation, scaling, and encoding are fitted on training data only to avoid data leakage.

In [ ]:
df = df.copy()
df.columns = df.columns.str.strip()

# Normalize text values and recognize common missing-value markers.
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({"?": pd.NA, "": pd.NA})

df = df.drop_duplicates().reset_index(drop=True)

# Remove rows with a missing target; features are imputed below.
df = df.dropna(subset=["income"]).copy()

X = df.drop(columns=["income"])
y_raw = df["income"].astype(str).str.strip()

# Map the common labels to 0/1. The fallback handles other two-class label names.
label_map = {"<=50K": 0, "<=50K.": 0, ">50K": 1, ">50K.": 1}
if set(y_raw.unique()).issubset(set(label_map)):
    y = y_raw.map(label_map).astype("int64")
else:
    labels = sorted(y_raw.unique())
    if len(labels) != 2:
        raise ValueError(f"Expected a binary income target, but found: {labels}")
    y = y_raw.map({labels[0]: 0, labels[1]: 1}).astype("int64")
    print("Target mapping used:", {labels[0]: 0, labels[1]: 1})

# Identify numeric and categorical columns.
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

# Compatibility across scikit-learn versions.
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot)
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder="drop")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
X_test = preprocessor.transform(X_test_raw).astype("float32")
y_train_np = y_train.to_numpy(dtype="float32")
y_test_np = y_test.to_numpy(dtype="int64")

print("Training matrix:", X_train.shape)
print("Testing matrix:", X_test.shape)
print("Training class balance:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

## 5. Define evaluation metrics
Accuracy alone can hide poor performance on a minority class. We also report precision, recall, F1-score, and ROC-AUC.

In [ ]:
def evaluate_predictions(name, y_true, probabilities):
    probabilities = np.asarray(probabilities).reshape(-1)
    predictions = (probabilities >= 0.5).astype("int64")
    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1-score": f1_score(y_true, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probabilities)
    }
    print(f"\n{name} classification report:")
    print(classification_report(y_true, predictions, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_true, predictions)
    plt.title(f"{name} - Confusion Matrix")
    plt.tight_layout()
    plt.show()
    return results

## 6. TensorFlow model
Use a feed-forward neural network for binary classification. The output is a single sigmoid probability; binary cross-entropy is the loss function. Early stopping helps reduce overfitting.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

tf_model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.20),
    layers.Dense(1, activation="sigmoid")
])
tf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)
tf_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )
]
tf_history = tf_model.fit(
    X_train, y_train_np,
    validation_split=0.15,
    epochs=15,
    batch_size=256,
    callbacks=tf_callbacks,
    verbose=1
)

tf_prob = tf_model.predict(X_test, batch_size=512, verbose=0).ravel()
tf_results = evaluate_predictions("TensorFlow", y_test_np, tf_prob)
print("TensorFlow results:", tf_results)

## 7. PyTorch model
Implement a similar network architecture in PyTorch. `BCEWithLogitsLoss` combines the sigmoid operation and binary cross-entropy in a numerically stable way, so the model returns logits during training.

In [ ]:
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class IncomeNet(nn.Module):
    def __init__(self, input_dim, hidden1=128, hidden2=64, dropout=0.30):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout / 1.5),
            nn.Linear(hidden2, 1)
        )
    def forward(self, x):
        return self.network(x).squeeze(1)

# Create a validation set from the training data for early stopping.
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_np, test_size=0.15, random_state=SEED,
    stratify=y_train_np
)

def make_loader(X_array, y_array, batch_size=256, shuffle=False):
    ds = TensorDataset(
        torch.from_numpy(np.asarray(X_array, dtype=np.float32)),
        torch.from_numpy(np.asarray(y_array, dtype=np.float32))
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_tr, y_tr, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test_np)

def train_torch_model(model, train_loader, val_loader, epochs=15, lr=1e-3, patience=3):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss, best_state, wait = float("inf"), None, 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_loss_sum, train_count = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * len(yb)
            train_count += len(yb)

        model.eval()
        val_loss_sum, val_count = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                loss = criterion(model(xb), yb)
                val_loss_sum += loss.item() * len(yb)
                val_count += len(yb)

        train_loss = train_loss_sum / train_count
        val_loss = val_loss_sum / val_count
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch+1:02d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

torch_model = IncomeNet(X_train.shape[1])
torch_model, torch_history = train_torch_model(
    torch_model, train_loader, val_loader, epochs=15, lr=1e-3
)

torch_model.eval()
torch_probs = []
with torch.no_grad():
    for xb, _ in test_loader:
        logits = torch_model(xb.to(device))
        torch_probs.extend(torch.sigmoid(logits).cpu().numpy())
torch_probs = np.asarray(torch_probs)
torch_results = evaluate_predictions("PyTorch", y_test_np, torch_probs)
print("PyTorch results:", torch_results)

## 8. Compare TensorFlow and PyTorch
The models use the same train/test split and the same fitted preprocessing, making the evaluation more comparable. The comparison table is populated from actual runs.

In [ ]:
comparison = pd.DataFrame([tf_results, torch_results]).set_index("Model")
display(comparison.round(4))

comparison[["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1)
)
plt.title("TensorFlow vs PyTorch performance")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

best_f1_model = comparison["F1-score"].idxmax()
best_auc_model = comparison["ROC-AUC"].idxmax()
print("Higher F1-score in this run:", best_f1_model)
print("Higher ROC-AUC in this run:", best_auc_model)

## 9. Optional: Optuna hyperparameter tuning (PyTorch)
This small search is deliberately limited to keep runtime manageable. It tunes hidden-layer width, dropout, learning rate, and batch size using validation F1-score. Increase `n_trials` only if time and compute resources allow. Test data is not used for tuning.

In [ ]:
import optuna
from sklearn.metrics import f1_score

def objective(trial):
    hidden1 = trial.suggest_categorical("hidden1", [64, 128])
    hidden2 = trial.suggest_categorical("hidden2", [32, 64])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [128, 256])

    trial_train_loader = make_loader(X_tr, y_tr, batch_size=batch_size, shuffle=True)
    trial_val_loader = make_loader(X_val, y_val, batch_size=batch_size)
    model = IncomeNet(X_train.shape[1], hidden1=hidden1, hidden2=hidden2, dropout=dropout)
    model, _ = train_torch_model(
        model, trial_train_loader, trial_val_loader, epochs=6, lr=lr, patience=2
    )

    model.eval()
    val_probs = []
    with torch.no_grad():
        for xb, _ in trial_val_loader:
            val_probs.extend(torch.sigmoid(model(xb.to(device))).cpu().numpy())
    val_preds = (np.asarray(val_probs) >= 0.5).astype(int)
    return f1_score(y_val.astype(int), val_preds, zero_division=0)

# Run only a few trials for a quick baseline comparison.
study = optuna.create_study(direction="maximize", study_name="income_prediction_tuning")
study.optimize(objective, n_trials=5, show_progress_bar=True)

print("Best validation F1:", study.best_value)
print("Best parameters:", study.best_params)
display(study.trials_dataframe().sort_values("value", ascending=False).head())

## 10. Train the tuned PyTorch model and evaluate on the test set
After selecting hyperparameters using validation data, retrain using the selected settings and evaluate once on the held-out test set.

In [ ]:
best = study.best_params
best_batch = best["batch_size"]
best_train_loader = make_loader(X_tr, y_tr, batch_size=best_batch, shuffle=True)
best_val_loader = make_loader(X_val, y_val, batch_size=best_batch)

tuned_model = IncomeNet(
    X_train.shape[1],
    hidden1=best["hidden1"],
    hidden2=best["hidden2"],
    dropout=best["dropout"]
)
tuned_model, tuned_history = train_torch_model(
    tuned_model, best_train_loader, best_val_loader,
    epochs=15, lr=best["lr"], patience=3
)

tuned_model.eval()
tuned_probs = []
with torch.no_grad():
    for xb, _ in test_loader:
        tuned_probs.extend(torch.sigmoid(tuned_model(xb.to(device))).cpu().numpy())
tuned_probs = np.asarray(tuned_probs)
tuned_results = evaluate_predictions("PyTorch + Optuna", y_test_np, tuned_probs)

final_comparison = pd.DataFrame([tf_results, torch_results, tuned_results]).set_index("Model")
display(final_comparison.round(4))

## 11. Findings and conclusion (complete after running)
Use the real metric values printed above to complete this section. Do not invent values.

- **Dataset and preprocessing:** The dataset contains census-related demographic and employment features. Missing values were imputed; numeric features were scaled; categorical features were one-hot encoded. The target `income` was converted into a binary label.
- **Evaluation:** Compare accuracy, precision, recall, F1-score, and ROC-AUC. F1-score balances precision and recall; ROC-AUC evaluates how well the model ranks the two classes across thresholds.
- **TensorFlow vs PyTorch:** Both frameworks can implement similar feed-forward neural networks. Differences in results can come from initialization, training details, and optimization rather than the framework alone.
- **Optuna:** Compare the tuned model's test metrics with the original PyTorch model. Tuning improves performance only if the measured results support that conclusion.
- **Limitations:** The Adult dataset may contain social and historical biases. Predictive performance does not imply that income prediction should be used for high-impact decisions about individuals.

### Final result summary
Fill this table from the `final_comparison` output after running the notebook.

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| TensorFlow | — | — | — | — | — |
| PyTorch | — | — | — | — | — |
| PyTorch + Optuna | — | — | — | — | — |

**Conclusion template:** “In this experiment, ______ achieved the highest F1-score, while ______ achieved the highest ROC-AUC. After Optuna tuning, the PyTorch model’s F1-score ______ compared with the untuned model. Therefore, based on the selected evaluation metric, ______ performed best on the held-out test set.”